In [4]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/subscriptions.duckdb')

con.execute("""
    CREATE OR REPLACE TABLE subscriptions AS
    SELECT * FROM read_csv_auto('../data/subscriptions.csv')
""")

print("✅ Database ready")
con.execute("SELECT COUNT(*), status FROM subscriptions GROUP BY status").df()

✅ Database ready


,count_star(),status
0,295,churned
1,2705,active


In [ ]:
mrr_df = con.execute("""
WITH monthly_mrr AS (
    SELECT
        STRFTIME(start_date, '%Y-%m') AS month,
        SUM(mrr) AS total_mrr,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN status='churned' THEN mrr ELSE 0 END) AS churned_mrr,
        SUM(CASE WHEN plan IN ('pro','enterprise') THEN mrr ELSE 0 END) AS premium_mrr
    FROM subscriptions
    GROUP BY 1
),
mrr_with_lag AS (
    SELECT *,
        LAG(total_mrr) OVER (ORDER BY month) AS prev_month_mrr
    FROM monthly_mrr
)
SELECT
    month,
    total_mrr,
    total_customers,
    total_mrr - prev_month_mrr AS net_new_mrr,
    ROUND(100.0*(total_mrr - prev_month_mrr)/NULLIF(prev_month_mrr,0),1) AS mrr_growth_pct,
    ROUND(100.0*churned_mrr/NULLIF(prev_month_mrr,0),2) AS churn_rate_pct,
    ROUND(100.0*premium_mrr/total_mrr,1) AS premium_mix_pct
FROM mrr_with_lag
ORDER BY month
""").df()

# Save for Streamlit later
mrr_df.to_csv('../data/mrr_monthly.csv', index=False)
mrr_df

OSError: Cannot save file into a non-existent directory: 'data'

In [ ]:
health_df = con.execute("""
WITH health_scores AS (
    SELECT
        customer_id, plan, mrr, industry,
        LEAST(tenure_months/12.0, 1.0)*25 AS tenure_score,
        LEAST(login_days_30/20.0, 1.0)*30 AS usage_score,
        LEAST(features_used/5.0, 1.0)*25  AS feature_score,
        GREATEST(20-(support_tickets*5), 0) AS support_score
    FROM subscriptions
    WHERE status = 'active'
)
SELECT *,
    ROUND(tenure_score+usage_score+feature_score+support_score,1) AS health_score,
    CASE
        WHEN tenure_score+usage_score+feature_score+support_score >= 75 THEN 'Healthy'
        WHEN tenure_score+usage_score+feature_score+support_score >= 50 THEN 'At Risk'
        ELSE 'Critical'
    END AS health_status
FROM health_scores
ORDER BY health_score ASC
""").df()

health_df.to_csv('../data/health_scores.csv', index=False)
print(health_df['health_status'].value_counts())
health_df.head(10)

In [ ]:
outlier_df = con.execute("""
WITH revenue_stats AS (
    SELECT
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY mrr) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY mrr) AS q3
    FROM subscriptions WHERE status='active'
),
bounds AS (
    SELECT q1, q3,
        q3-q1 AS iqr,
        q1 - 1.5*(q3-q1) AS lower_bound,
        q3 + 1.5*(q3-q1) AS upper_bound
    FROM revenue_stats
)
SELECT
    s.customer_id, s.mrr, s.plan, s.industry,
    ROUND(b.lower_bound,0) AS lower_bound,
    ROUND(b.upper_bound,0) AS upper_bound,
    CASE
        WHEN s.mrr > b.upper_bound THEN 'High — Upsell Opportunity'
        WHEN s.mrr < b.lower_bound THEN 'Low — Downgrade Risk'
    END AS outlier_flag
FROM subscriptions s
CROSS JOIN bounds b
WHERE s.status='active'
  AND (s.mrr > b.upper_bound OR s.mrr < b.lower_bound)
ORDER BY s.mrr DESC
""").df()

outlier_df.to_csv('../data/outliers.csv', index=False)
print(f"Outliers found: {len(outlier_df)}")
outlier_df.head(10)